<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet6%20(ANN)%20PBL/Praktikum%206%20(JS6)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=0fc68f12a92ba3c3071735c86d6e32ff03a3b6fd5cc831a5e040cb1af7fc1796
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 47.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=3f51d13e897d87c9b927ed4686a7f9d511ef0467432c10652e5d405cfe9c3804
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705

#Langkah 1: Persiapan Dataset & Fitur Numerik Cafe

In [ ]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# 1. Load dataset cafes.csv dan reviews.csv
cafes = pd.read_csv('cafes.csv')
reviews = pd.read_csv('reviews.csv')

# Agregasi data reviews per cafe
reviews_agg = reviews.groupby('cafe_id').agg(
    avg_review_rating=('rating', 'mean'),
    total_likes=('likes', 'sum')
).reset_index()

merged = pd.merge(cafes, reviews_agg, on='cafe_id', how='left').fillna(0)

# 2. Pilih hanya fitur numerik saja
feature_cols = ['latitude', 'longitude', 'rating', 'review_count', 'avg_review_rating', 'total_likes']
X = merged[feature_cols].values.astype(np.float32)

# 3. Standarisasi fitur menggunakan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors
print(f"Dataset berhasil dimuat dan dinormalisasi. Shape: {X_scaled.shape}")

Dataset berhasil dimuat dan dinormalisasi. Shape: (99, 6)


#Langkah 2: Exact Nearest Neighbor (Brute-Force)

In [ ]:
# ===============================
# Exact Nearest Neighbor (brute-force)
# ===============================
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start

print(f"Exact NN selesai dalam {time_exact:.4f} detik")
print("Contoh index tetangga terdekat (Item 0):", idx_exact[0][:5])

Exact NN selesai dalam 0.0503 detik
Contoh index tetangga terdekat (Item 0): [ 0 11 31  8 15]


#Langkah 3: Benchmark Annoy

In [ ]:
# ===============================
# Annoy
# ===============================
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')

for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)

index_annoy.build(10)  # 10 trees
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start

print(f"Annoy selesai dalam {time_annoy:.4f} detik")
print("Contoh index tetangga terdekat (Item 0):", idx_annoy[0][:5])

Annoy selesai dalam 0.0060 detik
Contoh index tetangga terdekat (Item 0): [0, 11, 31, 8, 6]


#Langkah 4: Benchmark HNSW (hnswlib)

In [ ]:
# ===============================
# HNSW
# ===============================
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(50)  # disesuaikan dengan skala 99 data cafe

idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start

print(f"HNSW selesai dalam {time_hnsw:.4f} detik")
print("Contoh index tetangga terdekat (Item 0):", idx_hnsw[0][:5])

HNSW selesai dalam 0.0156 detik
Contoh index tetangga terdekat (Item 0): [ 0 11 31  8  6]


#Langkah 5: Benchmark FAISS IVF

In [ ]:
# ===============================
# FAISS IVF
# ===============================
start = time.time()
nlist = 5  # disesuaikan dengan jumlah data cafe (99 baris)
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = IndexIVFFlat_custom = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], nlist, faiss.METRIC_L2)

# Training centroid (karena data < nlist * pelatihan, tangani dengan aman)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 2

dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start

print(f"FAISS IVF selesai dalam {time_faiss:.4f} detik")
print("Contoh index tetangga terdekat (Item 0):", idx_faiss[0][:5])

FAISS IVF selesai dalam 0.0251 detik
Contoh index tetangga terdekat (Item 0): [ 0 11 31  8  6]


#Langkah 6: Ringkasan Top-5 Neighbors & Analisis Praktikum

In [ ]:
# ===============================
# Ringkasan Top-5 Neighbors dari Item Pertama
# ===============================
print("\n=== Perbandingan Top-5 Neighbors (Cafe Pertama) ===")
print(f"Exact NN : {idx_exact[0][:5]}")
print(f"Annoy    : {idx_annoy[0][:5]}")
print(f"HNSW     : {idx_hnsw[0][:5]}")
print(f"FAISS    : {idx_faiss[0][:5]}")



=== Perbandingan Top-5 Neighbors (Cafe Pertama) ===
Exact NN : [ 0 11 31  8 15]
Annoy    : [0, 11, 31, 8, 6]
HNSW     : [ 0 11 31  8  6]
FAISS    : [ 0 11 31  8  6]
